In [0]:
%sql
use catalog cat_mini;

In [0]:
%sql
create or replace table gold.dim_customers as select 
customer_key , gender , continent from silver.customers;
    
create or replace table gold.dim_stores as select 
store_key,country from silver.stores;
    
create or replace table gold.dim_products as select 
product_key,category from silver.products;
  


In [0]:
%sql
create or replace table gold.fact_sales as select 
  s.order_number,s.line_item,s.order_date,s.delivery_date,
  s.customer_key,s.store_key,s.product_key,s.currency_code,
  s.quantity,p.unit_price_usd,
  (s.quantity * p.unit_price_usd) as revenue,
  datediff(s.delivery_date,s.order_date) as delivery_days,
  case 
    when s.store_key is null then 'Online'
    else 'Store'
  end as channel
from silver.sales s
left join silver.products p
on s.product_key = p.product_key;

In [0]:
%sql
create or replace table gold.cube as 
select * from gold.fact_sales left join gold.dim_customers using(customer_key) left join gold.dim_stores using(store_key) left join gold.dim_products using(product_key);

In [0]:
%sql
update gold.cube
set 
    delivery_days = COALESCE(delivery_days, 0),
    gender = COALESCE(gender, 'Unknown'),
    continent = COALESCE(continent, 'Unknown');

In [0]:
%sql
create or replace table gold.monthly_revenue as
select year(order_date) as year, month (order_date) as month, sum(revenue) as revenue from gold.cube group by year(order_date), month (order_date) order by year(order_date), month (order_date);

-- select * from gold.monthly_revenue;

In [0]:
%sql
create or replace view annual_total as select year , sum(revenue ) as total from gold.monthly_revenue group by year;
-- select * from annual_total;
create or replace table gold.top_3 as
select m.year,m.month, m.revenue , (m.revenue/a.total)*100 as percentage from gold.monthly_revenue m join annual_total a on m.year = a.year order by m.revenue desc limit 3;

-- select * from gold.top_3;

In [0]:
%sql
create or replace view category_monthly as
select year(order_date) as year, month (order_date) as month , category , sum(revenue) as cat_revenue from gold.cube group by year(order_date), month (order_date), category order by year(order_date), month (order_date);

-- select * from category_monthly;
create or replace view category_monthly_top as
select year, month , category, cat_revenue, revenue ,cat_revenue*100/revenue as percentage , row_number() over (partition by year,month order by cat_revenue desc) as rn from category_monthly join gold.top_3 using(year,month) order by year,month;

-- select * from category_monthly_top;

create or replace table gold.holiday_drivers as 
select year, month , category , cat_revenue , percentage from category_monthly_top where rn <= 3;

-- select * from gold.holiday_drivers;


In [0]:
%sql
create or replace table gold.delivery_performance as
select avg(delivery_days) as avg_days, count(*) as total_order from gold.cube;

-- select * from gold.delivery_performance;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.country_delivery AS
SELECT 
    country,
    AVG(delivery_days) AS avg_days,
    COUNT(*) AS order_count,
    PERCENTILE_CONT(0.5) 
        WITHIN GROUP (ORDER BY delivery_days) AS median_days
FROM gold.cube
GROUP BY country
-- having avg_days  > 0
ORDER BY avg_days DESC
LIMIT 5;

-- select * from gold.country_delivery;
 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.channel_aov AS
SELECT 
    continent,
 
    SUM(CASE WHEN channel = 'Online' THEN revenue END) 
    / COUNT(DISTINCT CASE WHEN channel = 'Online' THEN order_number END) 
    AS AOV_online,
 
    SUM(CASE WHEN channel = 'Store' THEN revenue END) 
    / COUNT(DISTINCT CASE WHEN channel = 'Store' THEN order_number END) 
    AS AOV_store,
 
    COUNT(DISTINCT CASE WHEN channel = 'Online' THEN order_number END) AS online_orders,
    COUNT(DISTINCT CASE WHEN channel = 'Store' THEN order_number END) AS store_orders
 
FROM gold.cube
GROUP BY continent;

UPDATE gold.channel_aov
SET 
    AOV_online = COALESCE(AOV_online, 0),
    AOV_store = COALESCE(AOV_store, 0),
    online_orders = COALESCE(online_orders, 0),
    store_orders = COALESCE(store_orders, 0);
-- SELECT * FROM gold.channel_aov;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.volume_leaders AS
WITH total AS (
    SELECT SUM(quantity) AS total_units FROM gold.cube
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY SUM(quantity) DESC) AS rank,
    category,
    SUM(quantity) AS units_sold,
    (SUM(quantity) / t.total_units) * 100 AS pct_of_total
FROM gold.cube
CROSS JOIN total t
GROUP BY category, t.total_units
ORDER BY units_sold DESC
LIMIT 5;

-- select * from gold.volume_leaders;
 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.revenue_leaders AS
WITH total AS (
    SELECT SUM(revenue) AS total_rev FROM gold.cube
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY SUM(revenue) DESC) AS rank,
    category,
    SUM(revenue) AS revenue_usd,
    (SUM(revenue) / t.total_rev) * 100 AS pct_of_total
FROM gold.cube
CROSS JOIN total t
GROUP BY category, t.total_rev
ORDER BY revenue_usd DESC
LIMIT 5;

-- select * from gold.revenue_leaders;
 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_profile AS
SELECT 
    continent,
    gender,
    COUNT(DISTINCT customer_key) AS customer_count,
    SUM(revenue) AS total_spend,
    SUM(revenue) / COUNT(DISTINCT customer_key) AS avg_spend
FROM gold.cube
GROUP BY continent, gender ORDER BY continent;


-- select * from gold.customer_profile;
 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_loyalty AS
WITH orders AS (
    SELECT 
        customer_key,
        COUNT(DISTINCT order_number) AS order_count
    FROM gold.cube
    GROUP BY customer_key
)
SELECT 
    c.continent,
    COUNT(*) AS unique_customers,
    COUNT(CASE WHEN o.order_count >= 2 THEN 1 END) AS repeat_customers,
    (COUNT(CASE WHEN o.order_count >= 2 THEN 1 END) * 100.0 / COUNT(*)) AS repeat_rate
FROM orders o
JOIN gold.cube c 
    ON o.customer_key = c.customer_key
GROUP BY c.continent;

-- select * from gold.customer_loyalty;